# Coinstore API Testing

In [ ]:
import hashlib
import hmac
import json
import math
import time
from typing import Dict, Any

import pandas as pd
import requests

## Configuration

In [ ]:
base_url = "https://api.coinstore.com"
symbol = "GGEZ1USDT"
symbol_lower = symbol.lower()

In [ ]:
# Accounts — uncomment the one you want to use

# Default account
api_key = "xx"
secret_key = "xx"

# MUTAZ account
# api_key = "xx"
# secret_key = "xx"

# MOHD account
# api_key = "xx"
# secret_key = "xx"

## Helper Functions

In [ ]:
def parse_params_to_str(params: Dict[str, Any]) -> str:
    return "?" + "&".join(f"{k}={v}" for k, v in params.items())


def generate_headers(data: dict, is_get_request: bool = False) -> Dict[str, str]:
    expires = int(time.time() * 1000)
    expires_key = str(math.floor(expires / 30000)).encode("utf-8")
    key = hmac.new(secret_key.encode("utf-8"), expires_key, hashlib.sha256).hexdigest().encode("utf-8")
    if is_get_request:
        payload = parse_params_to_str(data)[1:]
    else:
        payload = json.dumps(data)
    signature = hmac.new(key, payload.encode("utf-8"), hashlib.sha256).hexdigest()
    return {
        "X-CS-APIKEY": api_key,
        "X-CS-SIGN": signature,
        "X-CS-EXPIRES": str(expires),
        "exch-language": "en_US",
        "Content-Type": "application/json",
        "Accept": "*/*",
        "Connection": "keep-alive",
    }


def api_post(end_point: str, data: dict) -> dict:
    response = requests.post(
        base_url + end_point,
        headers=generate_headers(data),
        data=json.dumps(data).encode("utf-8"),
    )
    response.raise_for_status()
    return response.json()


def api_get(end_point: str, data: dict = None) -> dict:
    data = data or {}
    response = requests.get(
        base_url + end_point,
        headers=generate_headers(data, True),
        params=data,
    )
    response.raise_for_status()
    return response.json()


def pp(data):
    """Pretty print JSON."""
    print(json.dumps(data, indent=4))

## Account Balance

In [ ]:
balances = api_post("/api/spot/accountList", {})["data"]
for asset in balances:
    status = "Available" if asset["typeName"] != "FROZEN" else "Frozen"
    print(f"Asset: {asset['currency']} , {status}: {asset['balance']}")

## Active Orders

In [ ]:
active_orders = api_get("/api/trade/order/active", {"code": symbol})
pp(active_orders)

In [ ]:
# v2 endpoint — returns all active orders across symbols
all_active = api_get("/api/v2/trade/order/active")
pp(all_active)

## Place Order

In [ ]:
order_data = {
    "ordPrice": "0.0877",
    "ordQty": "24.61",
    "symbol": symbol,
    "side": "SELL",
    "ordType": "LIMIT",
}
result = api_post("/api/trade/order/place", order_data)
pp(result)

## Cancel Orders

### cancel all active orders

In [ ]:
# Cancel all active orders for the symbol
orders_to_cancel = api_get("/api/trade/order/active", {"code": symbol})

for order in orders_to_cancel["data"]:
    data = {"symbol": symbol_lower, "orderId": order["ordId"]}
    result = api_post("/api/trade/order/cancel", data)
    pp(result)
    time.sleep(1)

### cancel 1 order

In [ ]:
order_id = "2604110953500204290"
data = {"symbol": symbol_lower, "orderId": order_id}
result = api_post("/api/trade/order/cancel", data)
pp(result)

## Trade History

In [ ]:
matches = api_get("/api/trade/match/accountMatches", {
    "symbol": symbol_lower,
    "pageNum": 1,
    "pageSize": 5,
})

for trade in matches["data"]:
    trade["price"] = trade["execAmt"] / trade["execQty"]

pp(matches)

## Order Info

In [ ]:
order_id = "1852198375915654"
order_info = api_get("/api/trade/order/orderInfo", {"ordId": order_id})
pp(order_info)

## Market Data (Tickers)

In [ ]:
# All ticker prices (public, no auth needed)
response = requests.get(base_url + "/api/v1/ticker/price")
pp(response.json())

In [ ]:
# All market tickers with details
tickers = api_get("/api/v1/market/tickers")
pp(tickers)

## Generate Order Book

In [ ]:
orders = [
    # Sell orders
    {"ordQty": "5000.00", "ordPrice": str(0.08775), "side": "SELL"},
    {"ordQty": "5000.00", "ordPrice": str(0.0925), "side": "SELL"},
    {"ordQty": "5000.00", "ordPrice": str(0.092), "side": "SELL"},
    {"ordQty": "5000.00", "ordPrice": str(0.0915), "side": "SELL"},
    {"ordQty": "5000.00", "ordPrice": str(0.091), "side": "SELL"},
    {"ordQty": "10000.0", "ordPrice": str(0.0905), "side": "SELL"},
    {"ordQty": "10000.0", "ordPrice": str(0.09), "side": "SELL"},
    {"ordQty": "10000.0", "ordPrice": str(0.0895), "side": "SELL"},
    {"ordQty": "12000.0", "ordPrice": str(0.089), "side": "SELL"},
    {"ordQty": "14000.0", "ordPrice": str(0.0885), "side": "SELL"},
    {"ordQty": "16000.0", "ordPrice": str(0.088), "side": "SELL"},
    # Buy orders
    {"ordQty": "5000.00", "ordPrice": str(0.08725), "side": "BUY"},
    {"ordQty": "5000.00", "ordPrice": str(0.08675), "side": "BUY"},
    {"ordQty": "5000.00", "ordPrice": str(0.08625), "side": "BUY"},
    {"ordQty": "5000.00", "ordPrice": str(0.08575), "side": "BUY"},
    {"ordQty": "5000.00", "ordPrice": str(0.08525), "side": "BUY"},
    {"ordQty": "10000.0", "ordPrice": str(0.08475), "side": "BUY"},
    {"ordQty": "10000.0", "ordPrice": str(0.08425), "side": "BUY"},
    {"ordQty": "10000.0", "ordPrice": str(0.08375), "side": "BUY"},
    {"ordQty": "12000.0", "ordPrice": str(0.08325), "side": "BUY"},
    {"ordQty": "14000.0", "ordPrice": str(0.08275), "side": "BUY"},
    {"ordQty": "16000.0", "ordPrice": str(0.08225), "side": "BUY"},
]

for order in orders:
    order_data = {
        "ordPrice": order["ordPrice"],
        "ordQty": order["ordQty"],
        "symbol": symbol,
        "side": order["side"],
        "ordType": "LIMIT",
    }
    result = api_post("/api/trade/order/place", order_data)
    pp(result)
    time.sleep(1)

In [ ]:
for order in orders:
    order_data = {
        "ordPrice": order["ordPrice"],
        "ordQty": order["ordQty"],
        "symbol": symbol,
        "side": order["side"],
        "ordType": "LIMIT",
    }
    result = api_post("/api/trade/order/place", order_data)
    pp(result)
    time.sleep(1)